#Experiment 5: Subword Tokenization and POS Tagging
##5.1 BPE Tokenization

In [3]:
#a. With Pre-Trained Model
from transformers import GPT2Tokenizer

tokenizer_bpe_pre = GPT2Tokenizer.from_pretrained('gpt2')

with open('/content/input_sub_word_data.txt',
          'r', encoding='utf-8') as f:
    sample_text = f.read().strip().split('\n')[0]

print('--- BPE Pretrained ---')

encoded_bpe = tokenizer_bpe_pre.encode(sample_text)
tokens_bpe = tokenizer_bpe_pre.convert_ids_to_tokens(encoded_bpe)

print("Input:", sample_text)
print("\nTokens and Token IDs:")

for t, i in zip(tokens_bpe[:20], encoded_bpe[:20]):
    print(f'{t}: {i}')

--- BPE Pretrained ---
Input: Natural language processing is a branch of artificial intelligence that

Tokens and Token IDs:
Natural: 35364
Ġlanguage: 3303
Ġprocessing: 7587
Ġis: 318
Ġa: 257
Ġbranch: 8478
Ġof: 286
Ġartificial: 11666
Ġintelligence: 4430
Ġthat: 326


In [4]:
# b. Without pretrained model (Train from scratch)

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

file_path = '/content/input_sub_word_data.txt'

tokenizer_bpe_scratch = Tokenizer(BPE(unk_token='[UNK]'))
tokenizer_bpe_scratch.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=1000,
    special_tokens=['[UNK]']
)

# Train tokenizer on the dataset
tokenizer_bpe_scratch.train([file_path], trainer)

print('\n--- BPE From Scratch ---')
print('Input:', sample_text)

encoded_scratch = tokenizer_bpe_scratch.encode(sample_text)

print('\nTokens and Token IDs:')
for t, i in zip(encoded_scratch.tokens[:20], encoded_scratch.ids[:20]):
    print(f'{t}: {i}')


--- BPE From Scratch ---
Input: Natural language processing is a branch of artificial intelligence that

Tokens and Token IDs:
Natural: 355
language: 138
processing: 188
is: 78
a: 20
branch: 874
of: 85
artificial: 920
intelligence: 918
that: 262


##5.2 SentencePiece Tokenization

In [5]:
# a. Using pretrained model (T5 uses SentencePiece)
from transformers import T5Tokenizer
tokenizer_sp_pre = T5Tokenizer.from_pretrained('t5-small', legacy=False)

print('\n--- SentencePiece Pretrained ---')
encoded_sp = tokenizer_sp_pre.encode(sample_text)
tokens_sp = tokenizer_sp_pre.convert_ids_to_tokens(encoded_sp)
for t, i in zip(tokens_sp[:20], encoded_sp[:20]):
    print(f'{t}: {i}')

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]


--- SentencePiece Pretrained ---
▁Natural: 6869
▁language: 1612
▁processing: 3026
▁is: 19
▁: 3
a: 9
▁branch: 6421
▁of: 13
▁artificial: 7353
▁intelligence: 6123
▁that: 24
</s>: 1


In [6]:
# b. Without pretrained model (Train from scratch)

!pip install -q sentencepiece

import sentencepiece as spm

file_path = '/content/input_sub_word_data.txt'

# Train SentencePiece BPE tokenizer from scratch
spm.SentencePieceTrainer.train(
    input=file_path,
    model_prefix='m_sp',
    model_type='bpe',
    vocab_size=500,
    user_defined_symbols=[]
)

# Load trained tokenizer
sp = spm.SentencePieceProcessor(model_file='m_sp.model')

# Read sample sentence
with open(file_path, 'r', encoding='utf-8') as f:
    sample_text = next(line.strip() for line in f if line.strip())

print('\n--- SentencePiece BPE From Scratch ---')
print('Input:', sample_text)

# Generate tokens and IDs
encoded_sp_scratch = sp.encode_as_ids(sample_text)
tokens_sp_scratch = sp.encode_as_pieces(sample_text)

print('\nTokens and Token IDs:')
for t, i in zip(tokens_sp_scratch[:20], encoded_sp_scratch[:20]):
    print(f'{t}: {i}')


--- SentencePiece BPE From Scratch ---
Input: Natural language processing is a branch of artificial intelligence that

Tokens and Token IDs:
▁N: 213
atural: 174
▁language: 118
▁processing: 178
▁is: 84
▁a: 7
▁b: 32
r: 461
an: 15
ch: 210
▁of: 58
▁ar: 399
t: 460
ific: 327
ial: 384
▁int: 156
el: 61
l: 465
ig: 298
ence: 153


##5.3 POS Tagging (Short Sentence)

In [7]:
sentence = 'The young student is reading an interesting book in the library.'

# a. Using spaCy
!pip install -q spacy
!python -m spacy download en_core_web_sm -q

import spacy

nlp = spacy.load('en_core_web_sm')
doc = nlp(sentence)

print('\n--- POS Tagging (spaCy) ---')
print(f'{"Token":<15} | {"POS Tag":<10} | {"Description"}')
print('-' * 50)

for token in doc:
    print(f'{token.text:<15} | {token.pos_:<10} | {spacy.explain(token.pos_)}')


# b. Using NLTK
!pip install -q nltk

import nltk

nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

tokens_nltk = nltk.word_tokenize(sentence)
pos_tags_nltk = nltk.pos_tag(tokens_nltk)

print('\n--- POS Tagging (NLTK) ---')
print(f'{"Token":<15} | {"POS Tag":<10} | {"Description"}')
print('-' * 50)

for token, tag in pos_tags_nltk:
    print(f'{token:<15} | {tag:<10} | {tag}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 58.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.

--- POS Tagging (spaCy) ---
Token           | POS Tag    | Description
--------------------------------------------------
The             | DET        | determiner
young           | ADJ        | adjective
student         | NOUN       | noun
is              | AUX        | auxiliary
reading         | VERB       | verb
an              | DET        | determiner
interesting     | ADJ        | adjective
book            | NOUN       | noun
in              | ADP        | adposition
the             | DET        | determiner
library         | NOUN       | noun
.               | PUNCT  

##5.4 POS Tagging with Frequency (Large File)

In [9]:

from collections import Counter

with open('/content/input_sub_word_data.txt', 'r', encoding='utf-8') as f:
    large_text = f.read()

# Process first 5000 chars for demonstration
doc_large = nlp(large_text[:5000])

# Count frequencies of (Token, POS)
pos_freq = Counter([(token.text, token.pos_, spacy.explain(token.pos_)) for token in doc_large if not token.is_space])

print('\n--- POS Tagging with Frequency (spaCy) ---')
print(f'{"Token":<15} | {"POS Tag":<10} | {"Frequency":<10} | {"Description"}')
print('-'*65)
for (token, pos, desc), freq in pos_freq.most_common(20):
    print(f'{token:<15} | {pos:<10} | {freq:<10} | {desc}')


--- POS Tagging with Frequency (spaCy) ---
Token           | POS Tag    | Frequency  | Description
-----------------------------------------------------------------
,               | PUNCT      | 52         | punctuation
.               | PUNCT      | 52         | punctuation
the             | DET        | 26         | determiner
and             | CCONJ      | 21         | coordinating conjunction
a               | DET        | 19         | determiner
can             | AUX        | 15         | auxiliary
of              | ADP        | 13         | adposition
is              | AUX        | 11         | auxiliary
language        | NOUN       | 10         | noun
be              | AUX        | 10         | auxiliary
-               | PUNCT      | 10         | punctuation
as              | ADP        | 10         | adposition
words           | NOUN       | 9          | noun
word            | NOUN       | 9          | noun
BPE             | PROPN      | 9          | proper noun
may         

##5.5 Write a Python program to train a Word2Vec CBOW model using the given text corpus. Display the vector representation of a selected word and find the three most similar words.

In [12]:
# 5.5 Word2Vec CBOW Model

!pip install -q gensim nltk

from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Read corpus from Google Drive
file_path = '/content/corpus.txt'

with open(file_path, 'r', encoding='utf-8') as f:
    corpus = [line.strip() for line in f if line.strip()]

# Preprocess corpus
sentences = [word_tokenize(line.lower()) for line in corpus]

# Train Word2Vec CBOW model
model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=1,
    sg=0,       # CBOW
    workers=4,
    epochs=100
)

# Select a word
word = 'learning'

# Display vector representation
print("--- Word2Vec CBOW ---")
print(f"\nVector representation of '{word}':")
print(model.wv[word])

# Find three most similar words
print(f"\nThree most similar words to '{word}':")
for similar_word, similarity in model.wv.most_similar(word, topn=3):
    print(f"{similar_word}: {similarity:.4f}")

--- Word2Vec CBOW ---

Vector representation of 'learning':
[-0.00098594  0.00112105  0.00565675  0.01024967 -0.00955596 -0.00769141
  0.00759734  0.01018673 -0.00637803 -0.00492978  0.00770736 -0.00256769
 -0.00424793  0.0069163  -0.00446264 -0.00198138  0.00386623  0.00142034
 -0.00911312 -0.01144711  0.00789648  0.0057392   0.00779579  0.00064744
  0.00632103 -0.00275947 -0.00169939  0.00582666 -0.00788172 -0.0038764
 -0.0067547  -0.00166469  0.00999033 -0.00834166 -0.00283856 -0.00129005
  0.00863709 -0.00558688 -0.00026873 -0.00524384 -0.00860699  0.00442513
 -0.00944035 -0.00364466  0.00044686 -0.00069694 -0.00906448  0.00928218
  0.00571687  0.00936984 -0.00817094  0.00394769 -0.0044469   0.00047476
  0.00806996 -0.00434795  0.00513822 -0.00757574 -0.0036178   0.0094823
 -0.00172466  0.00015206 -0.00281103 -0.00746533 -0.00194382  0.00394365
 -0.00114578  0.00684925 -0.00367579  0.00206083  0.00564099  0.00924292
 -0.00073455 -0.00893281  0.00499723  0.00098492  0.00826689 -0.00

##5.6 Write a Python program to train a Skip-Gram Word2Vec model using the same corpus. Retrieve the vector for "language" and display the five most similar words.

In [13]:
# 5.6 Word2Vec Skip-Gram Model

!pip install -q gensim

from gensim.models import Word2Vec
import re

# Read corpus from Google Drive
file_path = '/content/corpus.txt'

with open(file_path, 'r', encoding='utf-8-sig') as f:
    corpus = [line.strip() for line in f if line.strip()]

if not corpus:
    raise ValueError("Corpus is empty. Check the file path and contents.")

# Tokenize corpus
sentences = [
    re.findall(r"\b[\w]+\b", line.lower())
    for line in corpus
]
sentences = [s for s in sentences if s]

# Create Skip-Gram Word2Vec model
model = Word2Vec(
    vector_size=100,
    window=5,
    min_count=1,
    sg=1,       # Skip-Gram
    workers=1,
    seed=42
)

# Build vocabulary
model.build_vocab(sentences)

# Train model
model.train(
    sentences,
    total_examples=model.corpus_count,
    epochs=100
)

# Retrieve vector for "language"
word = "language"

print("--- Word2Vec Skip-Gram ---")
print(f"\nVector representation of '{word}':")
print(model.wv[word])

# Display five most similar words
print(f"\nFive most similar words to '{word}':")
for similar_word, similarity in model.wv.most_similar(word, topn=5):
    print(f"{similar_word}: {similarity:.4f}")

--- Word2Vec Skip-Gram ---

Vector representation of 'language':
[ 9.6879741e-03  5.7166228e-03 -3.5500191e-03  9.5941853e-03
 -9.8624478e-05 -5.7861504e-05 -1.6242743e-03 -7.1518719e-03
  8.0690999e-03 -9.7836256e-03 -6.8937601e-03 -4.8516276e-03
 -4.0627941e-03 -7.0315152e-03 -9.3180696e-03  3.6536588e-03
  9.4249556e-03 -8.2934909e-03 -4.1903462e-03  3.4560458e-04
  8.0269445e-03  4.0432210e-03  5.3423280e-03  2.1309361e-03
 -1.0378674e-03 -6.4643337e-03  7.3149153e-03  6.1355047e-03
 -9.8001938e-03  4.5357146e-03 -8.5341930e-03  4.8362976e-03
  6.3542970e-03 -7.2827116e-03 -1.2297477e-03 -7.6462985e-03
  1.8419448e-03  7.6700030e-03 -8.7849163e-03 -1.8280445e-03
  9.6927369e-03 -4.2651063e-03 -7.0749670e-03 -7.1309577e-04
  2.7396006e-04  2.9715204e-03 -3.5514659e-03  9.6897548e-03
  8.6842524e-04 -4.7976985e-03  9.5639657e-03  8.2088411e-03
 -3.1859172e-04 -9.6796686e-03  2.3738204e-03  1.1600038e-03
 -7.1197832e-03  2.9545510e-03  6.8121618e-03 -7.5367391e-03
  5.1448541e-03 -7.1

##5.7 Train a Word2Vec model using the provided corpus and calculate the cosine similarity between selected word pairs.
##Input corpus: Use corpus.txt from Question 5.5.
##Calculate similarity for:
##1.machine and learning
##2.natural and language
##3.deep and learning


In [14]:
# 5.7 Word2Vec Cosine Similarity

!pip install -q gensim

from gensim.models import Word2Vec
import re

# Read corpus from Google Drive
file_path = '/content/corpus.txt'

with open(file_path, 'r', encoding='utf-8-sig') as f:
    corpus = [line.strip() for line in f if line.strip()]

if not corpus:
    raise ValueError("Corpus is empty. Check the file path.")

# Tokenize corpus
sentences = [
    re.findall(r"\b[\w]+\b", line.lower())
    for line in corpus
]
sentences = [s for s in sentences if s]

# Train Word2Vec model
model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=1,
    sg=0,       # CBOW
    workers=1,
    seed=42,
    epochs=100
)

# Selected word pairs
word_pairs = [
    ('machine', 'learning'),
    ('natural', 'language'),
    ('deep', 'learning')
]

# Calculate cosine similarity
print("--- Word2Vec Cosine Similarity ---")

for word1, word2 in word_pairs:
    similarity = model.wv.similarity(word1, word2)
    print(f"Similarity between '{word1}' and '{word2}': {similarity:.4f}")

--- Word2Vec Cosine Similarity ---
Similarity between 'machine' and 'learning': 0.1389
Similarity between 'natural' and 'language': 0.0618
Similarity between 'deep' and 'learning': 0.2513
